IMPORTS And PATHS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
import re, os, glob, subprocess, sys

from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/fast_datasets/'
CHARTS = BASE + 'charts/'
os.makedirs(CHARTS, exist_ok=True)

sb.set_style('whitegrid')
pd.options.display.float_format = '{:,.0f}'.format
print("Cell 1 done: setup ready")

Normalizer

In [ ]:
def normalize(name):
    """Standardize a company name so the same company matches across different files."""
    if pd.isna(name): return ""
    s = str(name).upper().strip()
    s = re.sub(r'[.,&\-/]', ' ', s)
    s = re.sub(r'\b(INC|LLC|LP|LLP|LTD|CORP|CORPORATION|COMPANY|CO|INCORPORATED|PLLC|LIMITED|L L C)\b', '', s)
    return re.sub(r'\s+', ' ', s).strip()

print("Cell 2 done")

Load and clean SBIR Awrds

In [ ]:
df = pd.read_csv(BASE + 'awards_search_1780524328.csv', low_memory=False)

df['amount']   = pd.to_numeric(df['Award Amount'], errors='coerce')
df['year']     = pd.to_numeric(df['Award Year'], errors='coerce').astype('Int64')
df['zip5']     = df['ZIP'].astype(str).str.split('-').str[0].str.extract(r'(\d{5})')[0]
df['name_key'] = df['Company Name'].apply(normalize)
df['award_end']= pd.to_datetime(df['Contract End Date'], errors='coerce')

tx10 = df[df['year'].between(2016, 2025)].copy()

print(f"Cell 3 done: {len(tx10):,} awards | {tx10['Company Name'].nunique()} companies | 2016–2025")
print("   columns check:", all(c in tx10.columns for c in ['amount','year','zip5','name_key']))

Build Company Table (one row per compnay)

In [ ]:
co = tx10.groupby('name_key').agg(
    company=('Company Name','first'),
    total_dollars=('amount','sum'),
    awards=('amount','size'),
    first_award_year=('year','min'),
    reached_II=('Phase', lambda s: 'Phase II' in set(s)),
    agency=('Agency', lambda s: s.mode().iloc[0]),
    uei=('UEI', lambda s: s.dropna().iloc[0] if s.notna().any() else None)
).reset_index()

print(f"Cell 4 done: company table built — {len(co)} companies")
print("   columns:", list(co.columns))

### ⏳ Correction: lifetime first-award year (fixes left-censoring)
**What this does & why.** The company table above sets `first_award_year` from `tx10` (2016–2025),
so a firm that first won *before* 2016 — including firms whose earlier awards merely *ended* around
2016 — is mislabeled as brand-new, which skews any "new vs. returning" or cohort read. This cell
recomputes each firm's **lifetime** first award year from the **full award history** (`df`, back to
1983) and shows how many firms that reclassifies.

In [ ]:
# [ADDED] lifetime first-award year from the FULL history (not just the 2016-2025 window)
lifetime_first = df.groupby('name_key')['year'].min().rename('first_year_lifetime')
co = co.drop(columns=[c for c in ['first_year_lifetime','new_to_sbir'] if c in co.columns], errors='ignore')
co = co.merge(lifetime_first, on='name_key', how='left')
co['new_to_sbir'] = co['first_year_lifetime'] >= 2016   # truly new to SBIR within the window
in_window_only = (co['first_award_year'] >= 2016).sum()
print(f"Truly new-to-SBIR firms (lifetime): {co['new_to_sbir'].sum()} of {len(co)}")
print(f"If judged only within 2016-2025 you'd wrongly call {in_window_only} firms 'new' "
      f"-- a {in_window_only - int(co['new_to_sbir'].sum())}-firm overcount.")

Initialize County Names

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pgeocode", "addfips"], check=True)
import pgeocode, addfips

# Pull company's home ZIP from awards_search
co_zip = (tx10.dropna(subset=['zip5'])
          .groupby('name_key')['zip5']
          .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else None)
          .reset_index())

# ZIP -> county name
uz = list(co_zip['zip5'].dropna().unique())
res = nomi.query_postal_code(uz)

county_col = next((c for c in res.columns if 'county' in c.lower()), None)
if county_col is None:
    raise ValueError(f"No county column. pgeocode returned: {list(res.columns)}")
print("   using pgeocode column:", county_col)

zip2county = dict(zip(res['postal_code'].astype(str), res[county_col]))
co_zip['county_name'] = co_zip['zip5'].astype(str).map(zip2county)

# FIPS code
af = addfips.AddFIPS()
def to_fips(c):
    if pd.isna(c): return None
    return af.get_county_fips(str(c).replace(' County',''), state='Texas')
co_zip['fips'] = co_zip['county_name'].apply(to_fips)

# attach to co
co = co.drop(columns=[c for c in ['county_name','fips'] if c in co.columns], errors='ignore')
co = co.merge(co_zip[['name_key','county_name','fips']], on='name_key', how='left')

print(f"Cell 5 done: county attached for {co['county_name'].notna().sum()} of {len(co)} companies")
print(co[['company','county_name','fips']].dropna().head(3).to_string(index=False))

Join Survival Status

In [ ]:
status = pd.read_csv(BASE + 'awardees_status_all.csv')

if 'name_key' not in status.columns:
    status['name_key'] = status['Company Name'].apply(normalize)

# drop old version first so re-running this cell stays clean
co = co.drop(columns=[c for c in ['still_active'] if c in co.columns], errors='ignore')
co = co.merge(status[['name_key','still_active']], on='name_key', how='left')
co['still_active'] = co['still_active'].fillna(False)

print(f"Cell 6 done: survival attached — {co['still_active'].sum()} of {len(co)} active "
      f"({co['still_active'].mean()*100:.0f}%)")

Load Company Contrtacts

In [ ]:
contract_files = glob.glob(BASE + 'Contracts_PrimeAwardSummaries*.csv')
print("Contract files found:")
for f in contract_files:
    print("  -", os.path.basename(f))

usa_list = [pd.read_csv(f, low_memory=False) for f in contract_files]
usa = pd.concat(usa_list, ignore_index=True) if usa_list else pd.DataFrame()

print(f"\nCell 7 done: {len(usa):,} contract rows from {len(contract_files)} file(s)")

Match Companies & Contracts by UEI

In [ ]:
contract_summary = (usa.groupby('recipient_uei')
                    .agg(contracts=('total_obligated_amount','size'),
                         contract_dollars=('total_obligated_amount','sum'))
                    .reset_index())

# drop old versions first so re-running stays clean
co = co.drop(columns=[c for c in ['contracts','contract_dollars','won_contract','recipient_uei'] if c in co.columns],
             errors='ignore')
co = co.merge(contract_summary, left_on='uei', right_on='recipient_uei', how='left')
co['won_contract'] = co['contracts'].notna()
co['contract_dollars'] = co['contract_dollars'].fillna(0)

print(f"Cell 8 done: {co['won_contract'].sum()} awardees won contracts "
      f"({co['won_contract'].mean()*100:.0f}%) — ${co['contract_dollars'].sum():,.0f} total")

Company to Contract Covnversion Gap table

In [ ]:
# Start from companies that have a county (i.e., were placed in Texas)
placed = co.dropna(subset=['county_name'])

# Statewide totals (the denominators)
total_awardees = len(placed)
total_winners  = placed['won_contract'].sum()

county_share = (placed.groupby('county_name')
                .agg(awardees=('company','size'),
                     winners=('won_contract','sum'))
                .reset_index())

# SHARE of statewide totals (the two bars)
county_share['pct_of_awardees'] = (county_share['awardees'] / total_awardees * 100).round(1)
county_share['pct_of_winners']  = (county_share['winners']  / total_winners  * 100).round(1)

# the gap: positive = under-converting (more award share than winner share)
county_share['gap'] = (county_share['pct_of_awardees'] - county_share['pct_of_winners']).round(1)

# include ALL counties; sort by award share so big ones lead, noisy tail trails
county_share = county_share.sort_values('pct_of_awardees', ascending=False)

print(f"Cell 9 done — {len(county_share)} counties | "
      f"{total_awardees} awardees, {total_winners} winners statewide")
print(county_share.to_string(index=False))

Conversion gap chart

In [ ]:
import numpy as np

# include all counties, but cap the x-axis labels so it stays readable;
# the long tail of tiny counties is still plotted, just grouped on the right
data = county_share.copy()
x = np.arange(len(data))
w = 0.42

fig, ax = plt.subplots(figsize=(max(12, len(data)*0.32), 6))
ax.bar(x - w/2, data['pct_of_awardees'], width=w, color='#90A4AE', label='% of all awardees')
ax.bar(x + w/2, data['pct_of_winners'],  width=w, color='#2E7D32', label='% of all contract-winners')

ax.set_xticks(x)
ax.set_xticklabels(data['county_name'], rotation=90, fontsize=7)
ax.set_ylabel('Share of Statewide Total (%)')
ax.set_title('Award Share vs Contract-Win Share by County (FY2024)\n'
             'Grey: share of awardee > Share of contract winner = county needing support')
ax.legend()

plt.tight_layout()
plt.savefig(CHARTS + 'county_share_funded_vs_converted.png', dpi=150, bbox_inches='tight')
plt.show()
print("Cell 10 done — saved county_share_funded_vs_converted.png")

In [ ]:
# [ADDED by Claude] Shared house style + CVD-validated agency palette for the sections below.
from matplotlib.ticker import FuncFormatter, PercentFormatter
plt.rcParams.update({'axes.spines.top':False,'axes.spines.right':False,'axes.axisbelow':True})
PAL={'DOD':'#0072B2','HHS':'#E69F00','NASA':'#009E73','NSF':'#CC79A7','DOE':'#D55E00','Other':'#7f7f7f'}
BLUE,ORANGE,GRAY='#0072B2','#E69F00','#9aa0a6'
TOPAG=['DOD','HHS','NASA','NSF','DOE']
usd=FuncFormatter(lambda v,_: f'${v/1e6:,.0f}M' if abs(v)>=1e6 else f'${v:,.0f}')
WIN=list(range(2016,2026))
def savec(fig,name): fig.savefig(CHARTS+name, dpi=150, bbox_inches='tight'); return fig
tx10['agency_grp']=tx10['Agency'].where(tx10['Agency'].isin(TOPAG),'Other')
print('style ready')

# ➕ Program overview & growth  *(added analysis)*
**What this section is for.** Before drilling into who wins and where, this establishes the size and
trajectory of Texas SBIR/STTR over the last decade — the headline numbers and the NC-SBTDC-style
"awards & awarded firms per year" view. **How to read it:** rising bars = a growing non-dilutive R&D
pipeline into Texas small businesses. *Source: SBIR/STTR award data (SBA), TX, 2016–2025.*

In [ ]:
# Headline + awards/firms/dollars per year
tot_usd=tx10['amount'].sum(); tot_aw=len(tx10); tot_fm=tx10['name_key'].nunique()
g=tx10.groupby('year')
aw=g.size().reindex(WIN,fill_value=0); fm=g['name_key'].nunique().reindex(WIN,fill_value=0)
dol=g['amount'].sum().reindex(WIN,fill_value=0)
base=tx10[tx10['year']==2016]['amount'].sum(); last=tx10[tx10['year']==2025]['amount'].sum()
print(f'2016-2025: ${tot_usd/1e9:.2f}B | {tot_aw:,} awards | {tot_fm:,} firms | $ growth 2025 vs 2016: {100*(last/base-1):+.0f}%')
import numpy as np
x=np.arange(len(WIN)); w=0.4
fig,ax=plt.subplots(figsize=(11,4.2))
b1=ax.bar(x-w/2,aw.values,w,label='Award count',color=BLUE)
b2=ax.bar(x+w/2,fm.values,w,label='# of firms',color=ORANGE)
ax.bar_label(b1,fontsize=8,color='#444'); ax.bar_label(b2,fontsize=8,color='#444')
ax.set_xticks(x); ax.set_xticklabels(WIN); ax.set_ylabel('Count')
ax.set_title('Texas SBIR/STTR awards and awarded firms per year',fontweight='bold',loc='left')
ax.legend(frameon=False,loc='upper left'); savec(fig,'add_awards_firms.png'); plt.show()
fig,ax=plt.subplots(figsize=(11,3.8))
bars=ax.bar(x,dol.values,color=BLUE,width=0.7)
ax.bar_label(bars,labels=[f'${v/1e6:.0f}M' for v in dol.values],fontsize=8,color='#444')
ax.set_xticks(x); ax.set_xticklabels(WIN); ax.yaxis.set_major_formatter(usd)
ax.set_title('Total award dollars per year',fontweight='bold',loc='left')
savec(fig,'add_dollars_year.png'); plt.show()

# ➕ Agency landscape & trends  *(added analysis)*
**What this section is for.** Shows *which federal agencies* drive Texas SBIR/STTR and how that mix is
changing — the "how are the agencies developing" question. **How to read it:** each line/bar is one
agency (top 5 named, the rest folded into a neutral *Other*); colors are fixed per agency.
*Source: SBIR/STTR award data (SBA), TX, 2016–2025.*

In [ ]:
# Agency awards over time + dollar share + SBIR/STTR
piv=tx10.groupby(['year','agency_grp']).size().unstack(fill_value=0).reindex(WIN,fill_value=0)
order=[a for a in TOPAG+['Other'] if a in piv.columns]
fig,ax=plt.subplots(figsize=(11,4.4))
for a in order: ax.plot(WIN,piv[a].values,marker='o',ms=4,lw=2.2,color=PAL[a],label=a)
ymax=float(piv[order].values.max()); gap=ymax*0.06; placed=[]
for a,val in sorted(((a,float(piv[a].values[-1])) for a in order),key=lambda t:t[1]):
    y=val if not placed or val-placed[-1][1]>=gap else placed[-1][1]+gap; placed.append((a,y))
for a,y in placed: ax.annotate(a,(WIN[-1],y),xytext=(8,0),textcoords='offset points',va='center',fontsize=9,fontweight='bold',color=PAL[a])
ax.set_xticks(WIN); ax.set_ylabel('Awards per year'); ax.set_xlim(WIN[0],WIN[-1]+0.8)
ax.set_title('SBIR/STTR awards by funding agency',fontweight='bold',loc='left')
savec(fig,'add_agency_lines.png'); plt.show()
share=tx10.groupby('agency_grp')['amount'].sum().sort_values()
share=share.reindex([a for a in ['Other','DOE','NSF','NASA','HHS','DOD'] if a in share.index])
fig,ax=plt.subplots(figsize=(9,3.4))
bars=ax.barh(share.index,share.values,color=[PAL[a] for a in share.index]); tot=share.sum()
for b,v in zip(bars,share.values): ax.text(v+tot*0.01,b.get_y()+b.get_height()/2,f'${v/1e6:.0f}M ({100*v/tot:.0f}%)',va='center',fontsize=9,color='#333')
ax.xaxis.set_major_formatter(usd); ax.set_xlim(0,tot*0.72)
ax.set_title('Share of award dollars by agency, 2016-2025',fontweight='bold',loc='left')
savec(fig,'add_agency_share.png'); plt.show()

# ➕ Recipients & concentration  *(added analysis)*
**What this section is for.** Who captures the money, and how concentrated is it? Uses your `co`
table so it stays consistent with your tiers. Includes a **Lorenz curve + Gini** (dollar
concentration), the **new-vs-returning** view built on the *lifetime* first-award fix above, and
self-reported **ownership** demographics. **How to read Lorenz:** the farther the blue curve bows
below the diagonal, the more concentrated the dollars. *Source: SBA award data, TX, 2016–2025.*

In [ ]:
import numpy as np
# Top 15 firms by awards
t=co.sort_values('awards').tail(15)
fig,ax=plt.subplots(figsize=(9,5.5)); bars=ax.barh(t['company'],t['awards'],color=BLUE)
for b,v in zip(bars,t['awards']): ax.text(v+3,b.get_y()+b.get_height()/2,f'{int(v)}',va='center',fontsize=9,color='#333')
ax.set_xlabel('SBIR/STTR awards, 2016-2025'); ax.set_title('Most prolific Texas firms (by award count)',fontweight='bold',loc='left')
savec(fig,'add_top_firms.png'); plt.show()
# Lorenz + Gini on total_dollars
v=np.sort(co['total_dollars'].values); n=len(v); cum=np.cumsum(v)/v.sum(); xx=np.arange(1,n+1)/n
gini=(2*np.sum(np.arange(1,n+1)*v)-(n+1)*np.sum(v))/(n*np.sum(v))
fig,ax=plt.subplots(figsize=(6,6))
ax.plot([0,1],[0,1],ls='--',color=GRAY,lw=1.5,label='Perfect equality')
ax.plot(xx,cum,color=BLUE,lw=2.4,label='Actual'); ax.fill_between(xx,cum,xx,color=BLUE,alpha=0.08)
ax.xaxis.set_major_formatter(PercentFormatter(1.0)); ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel('Cumulative share of firms'); ax.set_ylabel('Cumulative share of dollars')
ax.set_title(f'Award dollars are concentrated (Gini \u2248 {gini:.2f})',fontweight='bold',loc='left'); ax.legend(frameon=False,loc='upper left')
savec(fig,'add_lorenz.png'); plt.show()
top10=100*co['total_dollars'].sort_values(ascending=False).head(10).sum()/co['total_dollars'].sum()
print(f'Top 10 firms hold {top10:.0f}% of all TX SBIR/STTR dollars.')
# New vs returning (lifetime-correct)
rows=[]
for y in WIN:
    active=tx10[tx10['year']==y]['name_key'].unique()
    fy=co.set_index('name_key')['first_year_lifetime']
    new=sum(fy.get(f,y)==y for f in active); rows.append((y,new,len(active)-new))
nf=pd.DataFrame(rows,columns=['year','new','ret']).set_index('year')
fig,ax=plt.subplots(figsize=(11,4))
ax.bar(nf.index,nf['ret'],color=BLUE,label='Returning (won before)')
ax.bar(nf.index,nf['new'],bottom=nf['ret'],color=ORANGE,label='New to SBIR (lifetime)')
ax.set_xticks(WIN); ax.set_ylabel('Distinct firms funded')
ax.set_title('New vs. returning awardees each year (lifetime-corrected)',fontweight='bold',loc='left'); ax.legend(frameon=False,loc='upper left')
savec(fig,'add_new_returning.png'); plt.show()
# Ownership demographics
flags=[('Women Owned','Woman-owned'),('Socially Economically Disadvantaged','Disadvantaged'),('Hubzone Owned','HUBZone')]
labels=[l for _,l in flags]
aws=[100*tx10[c].astype(str).str.upper().eq('Y').mean() for c,_ in flags]
uss=[100*tx10.loc[tx10[c].astype(str).str.upper().eq('Y'),'amount'].sum()/tx10['amount'].sum() for c,_ in flags]
x=np.arange(len(labels)); w=0.38; fig,ax=plt.subplots(figsize=(8,4))
b1=ax.bar(x-w/2,aws,w,color=GRAY,label='% of awards'); b2=ax.bar(x+w/2,uss,w,color=BLUE,label='% of dollars')
ax.bar_label(b1,fmt='%.1f%%',fontsize=9); ax.bar_label(b2,fmt='%.1f%%',fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel('Percent of TX total'); ax.legend(frameon=False)
ax.set_title('Awards to under-represented-owned firms',fontweight='bold',loc='left')
savec(fig,'add_demographics.png'); plt.show()

# ➕ Geography: unique firms by county  *(added analysis)*
**What this section is for.** Your annotation asked for **# of *unique firms* by county** (distinct
companies, not award rows). This reuses the `county_name` your pipeline already assigned via
pgeocode+addfips. **How to read it:** longer bars = more distinct SBIR/STTR *companies* headquartered
in that county. *Source: SBA award data + Census/pgeocode county mapping.*

In [ ]:
# Unique firms by county (guarded on county having been assigned)
if 'county_name' in co.columns and co['county_name'].notna().any():
    byc=(co.dropna(subset=['county_name']).groupby('county_name')
         .agg(firms=('name_key','nunique'), dollars=('total_dollars','sum')).reset_index()
         .sort_values('firms',ascending=True))
    top=byc.tail(15)
    fig,ax=plt.subplots(figsize=(9,5.5)); bars=ax.barh(top['county_name'],top['firms'],color=BLUE)
    for b,v in zip(bars,top['firms']): ax.text(v+0.3,b.get_y()+b.get_height()/2,f'{int(v)}',va='center',fontsize=9,color='#333')
    ax.set_xlabel('Distinct SBIR/STTR firms'); ax.set_title('Texas counties by number of unique SBIR/STTR firms',fontweight='bold',loc='left')
    savec(fig,'add_firms_by_county.png'); plt.show()
    print('Top counties by unique firms:'); print(byc.tail(8)[['county_name','firms']].to_string(index=False))
else:
    print('Run the county cells above first (co.county_name not populated).')

---
*Added sections above complement the survival + contract-conversion + maturity-tier analysis in the
original notebook. Still to add (need approved sources / national data): Phase I→II→III **funnel**
visual, **per-capita** county/state normalization (Census population), and the NC-style national
**state-ranking** table. — merged by Claude.*